# Walkthrough — from the lab's HC and GA to the final algorithms

**Problem.** $N$ children each have $C$ Boolean characteristics. Seat them in a circle to maximize the sum of Hamming distances between neighbours (the last child included).

**Assignment.** Solve it with a hill climber (HC) and a genetic algorithm (GA).

This notebook runs, on one instance and with the same time allowance for every algorithm:

1. the lab's HC and GA (the first versions),
2. the best classic version of HC,
3. an exact solver, which reveals how far those were from the optimum,
4. the enhancements that close the gap — each one a separate, switchable layer,
5. a final comparison, also at a size where the classic versions cannot run.

Each section matches one step of [`HISTORY.md`](HISTORY.md), which explains the concepts in more detail. The code lives in [`project/`](project/README.md); every version (an *era*) is also runnable from there with `python -m circle_optimization run ERA`.

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd() / "project"          # the full project lives in project/
if (PROJECT / "circle_optimization").is_dir() and str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from random import Random
from statistics import mean
from time import perf_counter

from circle_optimization.eras import ERAS, Problem, run_era

CHILDREN, CHARACTERISTICS, INSTANCE_SEED = 160, 5, 2027   # the benchmark's instance
TIME_LIMIT = 0.2                                          # seconds per run, the same for every algorithm
SEEDS = (1, 2, 3)                                         # search seeds, shared by all algorithms

problem = Problem.random(CHILDREN, CHARACTERISTICS, seed=INSTANCE_SEED)
problem.distances, problem.state    # build the shared representations before any timed run
print(problem.label, '| distinct profiles:', problem.state.m, 'of', 2 ** CHARACTERISTICS, 'possible')

N=160 C=5 seed=2027 | distinct profiles: 32 of 32 possible


In [2]:
SCORES = {}

def run(*names, label=None, **overrides):
    # Run each era on SEEDS with TIME_LIMIT; print mean and best score (and the gap once the optimum is known).
    optimum = globals().get('OPTIMUM')
    print(f"{'algorithm':<28} {'phase':<5} {'mean':>7} {'best':>5}" + (f" {'gap %':>7}" if optimum else ''))
    for name in names:
        scores = [run_era(name, problem, seed=seed, time_limit=TIME_LIMIT, **overrides)['score'] for seed in SEEDS]
        key = label or name
        SCORES[key] = scores
        line = f"{key:<28} {ERAS[name].phase:<5} {mean(scores):>7.1f} {max(scores):>5}"
        if optimum:
            line += f" {100 * (optimum - mean(scores)) / optimum:>7.2f}"
        print(line)

## 1. The lab's solution: HC and GA (`hc-legacy`, `ga`)

A circle is an order of child indices (child 0 fixed first, one canonical direction). Distances come from the precomputed $N \times N$ Hamming matrix.

- **Moves.** A *swap* exchanges two children; an *inversion* reverses a segment (the 2-opt move). A move changes at most four edges, so its effect is computed in $O(1)$ without building the new circle.
- **HC.** Propose a random move, keep it only if it strictly improves, restart from a random circle when stuck.
- **GA.** Tournament selection, order crossover (OX), one swap/inversion mutation per child, elitism, fitness cache.

In [3]:
run('hc-legacy', 'ga')

algorithm                    phase    mean  best
hc-legacy                    01      616.7   618
ga                           01      635.3   638


## 2. A better classic HC (`hc`, `hc-i`)

Measurements showed the first HC restarting searches that were still improving (a hard cap of 1,000 proposals). **Productive restarts** wait for 200 consecutive failures instead; `hc-i` also proposes 80% inversions, which suit circular tours.

In [4]:
run('hc', 'hc-i')

algorithm                    phase    mean  best
hc                           04      666.7   676
hc-i                         04      668.0   670


## 3. How good is that? The exact optimum (`exact`)

There are only $m \le 2^C$ distinct profiles (32 here), and children with the same profile are interchangeable. A circle is therefore described, up to relabelling, by an $m \times m$ table counting the neighbouring pairs between each two profiles. The exact solver optimizes that table with a MILP (degree $2 n_p$ per profile, connectivity cuts added lazily, LP bound, reduced-cost fixing) and turns the optimal table back into a circle with an Euler circuit. Its size does not depend on $N$.

In [5]:
start = perf_counter()
OPTIMUM = problem.optimum
print(f"proven optimum {OPTIMUM} (proof: {problem.exact['proof']}) in {1000 * (perf_counter() - start):.1f} ms\n")
for key, scores in SCORES.items():
    print(f"{key:<12} mean {mean(scores):6.1f}   gap {100 * (OPTIMUM - mean(scores)) / OPTIMUM:5.2f}%")

proven optimum 748 (proof: bound) in 77.9 ms

hc-legacy    mean  616.7   gap 17.56%
ga           mean  635.3   gap 15.06%
hc           mean  666.7   gap 10.87%
hc-i         mean  668.0   gap 10.70%


The classic versions stop well below the optimum (gaps above; about 10–18% on this instance in the recorded benchmark). From their final circles, moves on the profile table alone close most of that gap, and many of their edges join profile pairs that the LP proves sub-optimal: the searches miss *moves* and *information*, not time. The next sections add both, one layer at a time, without changing what HC and GA are.

## 4. Layer S2 — profile state (`hc-p`)

The circle is stored as a sequence of profile ids and distances come from the $m \times m$ table. Moves and decisions are unchanged, so with the same seed and budget `hc-p` follows exactly the same trajectory as `hc` — but with $O(N + m^2)$ memory instead of $O(N^2)$, and with profile identities visible to the next layers.

In [6]:
classic = run_era('hc', problem, seed=7, evaluation_budget=20_000)
layered = run_era('hc-p', problem, seed=7, evaluation_budget=20_000)
print('identical search:', all(classic[k] == layered[k] for k in ('score', 'evaluations', 'restarts', 'stop_reason')))
print(f"lookup cells: {problem.n ** 2:,} (N x N) vs {problem.state.m ** 2:,} (m x m)")

identical search: True
lookup cells: 25,600 (N x N) vs 1,024 (m x m)


## 5. Layer S1 — split-and-splice (`hc-p-splice`)

Cutting two edges `a–b` and `c–d` leaves two reconnections. The inversion uses `a–c, b–d`. The other, `a–d, b–c`, splits the circle into two loops — unless both loops contain a child of the **same profile**: exchanging those two children's successors joins the loops again at zero cost. Its score change is still $O(1)$.

In [7]:
from circle_optimization.classic.operators import inversion_delta
from circle_optimization.layered.splice import splice_delta

n, table = problem.n, problem.state.table
local = run_era('hc-p', problem, seed=1, time_limit=TIME_LIMIT)['sequence']
pairs = [(i, j) for i in range(1, n - 1) for j in range(i + 1, n)]
print('at a final hc-p circle:',
      sum(inversion_delta(local, table, i, j) > 0 for i, j in pairs), 'improving inversions,',
      sum(splice_delta(local, table, i, j) > 0 for i, j in pairs), 'improving splices')
run('hc-p-splice')

at a final hc-p circle: 112 improving inversions, 104 improving splices
algorithm                    phase    mean  best   gap %
hc-p-splice                  10      678.7   690    9.27


## 6. Layer S3 — LP reduced-cost guidance (`hc-p-rc`)

The LP relaxation of the exact model gives every profile pair a *reduced cost* $rc \le 0$, and every circle satisfies $\text{score} \le \text{LP bound} - \sum |rc|$ over its edges. Edges on pairs with $rc < 0$ provably cost points; optimal circles use only $rc = 0$ pairs. The guide breaks the worst edge of a small random tournament and re-attaches it to an $rc = 0$ partner (20% of proposals stay uniform).

In [8]:
from circle_optimization.layered.reduced_costs import ReducedCostGuide

guide = ReducedCostGuide.from_state(problem.state)
m = problem.state.m
allowed = sum(guide.reduced_costs[p][q] == 0 for p in range(m) for q in range(p + 1, m))
print(f"LP bound {guide.bound:.0f}; profile pairs an optimal circle may use: {allowed} of {m * (m - 1) // 2}")
for name in ('hc-p-splice', 'hc-p-rc'):
    sequence = run_era(name, problem, seed=1, time_limit=TIME_LIMIT)['sequence']
    print(f"  {name:<12} score {problem.state.score(sequence)}: proven distance to the bound sum|rc| = {guide.certificate(sequence):.0f}")
run('hc-p-rc')

LP bound 748; profile pairs an optimal circle may use: 125 of 496
  hc-p-splice  score 674: proven distance to the bound sum|rc| = 74
  hc-p-rc      score 738: proven distance to the bound sum|rc| = 10
algorithm                    phase    mean  best   gap %
hc-p-rc                      11      738.0   738    1.34


## 7. Layers S5 and S6 — propagation and greedy restarts (`hc-p-prop`, `hc-p-grasp`)

- **S5.** A move's gain depends only on the four profiles involved, so an accepted pattern is retried at its other occurrences. It helps on mid-size instances and costs time on very large ones (see `HISTORY.md`).
- **S6.** Random circles start at about half the optimum. GRASP restarts build every start with the randomized greedy construction used by the exact solver's heuristic; HC still does all the climbing.

In [9]:
run('hc-p-prop', 'hc-p-grasp')

algorithm                    phase    mean  best   gap %
hc-p-prop                    13      737.3   740    1.43
hc-p-grasp                   14      748.0   748    0.00


## 8. The GA side (`ga-e`, `ga-m`, `ga-e-seed`)

OX preserves the *order* of the parents, but the score depends on *adjacency*. The edge GA's genome is the profile edge table: a child keeps every edge its parents share, adds edges from their union, completes the degrees greedily and repairs connectivity. A separate, optional **memetic layer** improves 20% of the children by local search (`memetic=None` switches it off). Seeding puts 10% greedy constructions into the first population.

In [10]:
import numpy as np
from circle_optimization.layered.edge_genome import edge_crossover, genome_fitness, random_genome

generator = np.random.default_rng(0)
a, b = random_genome(problem.state, generator), random_genome(problem.state, generator)
child = edge_crossover(a, b, problem.state, generator, fill='greedy')
print('parents', genome_fitness(a, problem.state), genome_fitness(b, problem.state),
      '-> child', genome_fitness(child, problem.state),
      '| keeps every shared edge:', bool((child >= np.minimum(a, b)).all()))
run('ga-e', 'ga-m', 'ga-e-seed')
run('ga-m', label='ga-m without memetic layer', memetic=None)

parents 408 414 -> child 502 | keeps every shared edge: True
algorithm                    phase    mean  best   gap %
ga-e                         12      668.7   674   10.61
ga-m                         12      741.3   742    0.89
ga-e-seed                    14      748.0   748    0.00
algorithm                    phase    mean  best   gap %
ga-m without memetic layer   12      670.0   674   10.43


## 9. All versions side by side

In [11]:
print(f"{'algorithm':<28} {'gap %':>6}")
for key in sorted(SCORES, key=lambda key: -mean(SCORES[key])):
    gap = 100 * (OPTIMUM - mean(SCORES[key])) / OPTIMUM
    print(f"{key:<28} {gap:>6.2f}  {'#' * round(2 * gap)}")

algorithm                     gap %
hc-p-grasp                     0.00  
ga-e-seed                      0.00  
ga-m                           0.89  ##
hc-p-rc                        1.34  ###
hc-p-prop                      1.43  ###
hc-p-splice                    9.27  ###################
ga-m without memetic layer    10.43  #####################
ga-e                          10.61  #####################
hc-i                          10.70  #####################
hc                            10.87  ######################
ga                            15.06  ##############################
hc-legacy                     17.56  ###################################


## 10. Beyond the $N \times N$ matrix

At $N = 10{,}000$ the classic versions would need a $10^8$-cell distance matrix. The profile-based versions and the exact solver work on the $m \times m$ table instead.

In [12]:
large = Problem.random(10_000, 8, seed=INSTANCE_SEED)
start = perf_counter()
large_optimum = large.optimum
print(f"{large.label}: m = {large.state.m}, optimum {large_optimum} proven in {perf_counter() - start:.2f} s")
for name in ('hc-p', 'hc-p-rc', 'hc-p-grasp', 'ga-e-seed'):
    result = run_era(name, large, seed=1, time_limit=1.0)
    print(f"  {name:<12} {result['score']:>7}   gap {100 * (large_optimum - result['score']) / large_optimum:6.2f}%")

N=10000 C=8 seed=2027: m = 256, optimum 78850 proven in 0.70 s
  hc-p           57354   gap  27.26%
  hc-p-rc        70406   gap  10.71%
  hc-p-grasp     78656   gap   0.25%
  ga-e-seed      78726   gap   0.16%


## Going further

- [`HISTORY.md`](HISTORY.md) — every step and the concept behind it.
- [`project/README.md`](project/README.md) — the full project: package, tests, benchmark suite, records.
- [`project/guided_tour.ipynb`](project/guided_tour.ipynb) — every era, including the guided and ILS variants, and a layer-by-layer ablation.
- From `project/`: `python -m circle_optimization list`, `... compare --all --children 1000`, `python benchmark_guided_search.py --suite --no-classic --eras hc hc-p-grasp ga ga-e-seed --protocol time`.